# Practica 3 sobre los TRnasfomaciones de KNN.

**Nombres:** Miguel Vanegas y José Vanegas  
**Fecha:** 14/05/2026

## Cargar datos
# Fase1.1: Importacion de libreras y Cargar los datos,

In [17]:
import numpy as np
import pandas as pd
import copy
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


ruta =  "data/data.csv"

dfOriginal=pd.read_csv(ruta)

cData=copy.deepcopy(dfOriginal)

print(cData.shape)
print(cData.head(10))



(6, 5)
   sexo     ciudad colesterol  edad diabetes
0     1     Cuenca       bajo    18       no
1     2      Quito       alto    52       si
2     2  Guayaquil      medio    34       no
3     1       Loja       alto    61       si
4     2     Ambato      medio    45       no
5     1    Machala   muy alto    67       si


## Fase 1.2 Identificacion de cada variale y variable objetivo

In [18]:
var_numericas = ["sexo", "edad"]
var_nominales = ["ciudad"]
var_ordinales = ["colesterol"]
var_objetivo = "diabetes"

# Separar variables predictoras (X) y objetivo (y)
X = cData.drop(columns=["diabetes"])
y = cData["diabetes"].map({"no":0, "si": 1})

print("Variables numéricas:", var_numericas)
print("Variables categóricas nominales:", var_nominales)
print("Variables categóricas ordinales:", var_ordinales)
print("Variable objetivo:", var_objetivo)

Variables numéricas: ['sexo', 'edad']
Variables categóricas nominales: ['ciudad']
Variables categóricas ordinales: ['colesterol']
Variable objetivo: diabetes


## Fase 2 Transformacion de las variables 


In [19]:
# ## Orden que esta la varieble colesterol 
orden_var=[["bajo","medio","alto","muy alto"]]

preprocesador = ColumnTransformer(transformers=[
    ('cat_nom', OneHotEncoder(sparse_output=False, handle_unknown="ignore"), var_nominales),
    ('cat_ord', OrdinalEncoder(categories=orden_var), var_ordinales)
], remainder='passthrough') # <--- ¡ESTA ES LA MAGIA QUE FALTABA!

pipe_maestro = Pipeline(steps=[
    ('preprocesamiento', preprocesador),
    ('estandarizacion_total', StandardScaler()) # Estandariza todo de golpe
])

X_transformado = pipe_maestro.fit_transform(X)
# 5. RECONSTRUCCIÓN DE LA TABLA
nombres_nominales = pipe_maestro.named_steps['preprocesamiento'].named_transformers_['cat_nom'].get_feature_names_out(var_nominales)

# Como usamos 'passthrough', las numéricas salen al final, así que el orden de tus nombres es correcto:
columnas_finales = list(nombres_nominales) + var_ordinales + var_numericas

# Ahora sí, los datos (9 columnas) encajarán perfectamente con los nombres (9 nombres)
df_final = pd.DataFrame(data=X_transformado, columns=columnas_finales)

# --- IMPRESIÓN 2: LA TABLA ESTANDARIZADA ---
print("********** 2. TABLA ESTANDARIZADA (Todas las variables) **********")
print(df_final.round(4).head(6).to_string(index=False))
print("\n" + "="*60 + "\n")

********** 2. TABLA ESTANDARIZADA (Todas las variables) **********
 ciudad_Ambato  ciudad_Cuenca  ciudad_Guayaquil  ciudad_Loja  ciudad_Machala  ciudad_Quito  colesterol  sexo    edad
       -0.4472         2.2361           -0.4472      -0.4472         -0.4472       -0.4472     -1.5667  -1.0 -1.7085
       -0.4472        -0.4472           -0.4472      -0.4472         -0.4472        2.2361      0.5222   1.0  0.3538
       -0.4472        -0.4472            2.2361      -0.4472         -0.4472       -0.4472     -0.5222   1.0 -0.7380
       -0.4472        -0.4472           -0.4472       2.2361         -0.4472       -0.4472      0.5222  -1.0  0.8997
        2.2361        -0.4472           -0.4472      -0.4472         -0.4472       -0.4472     -0.5222   1.0 -0.0708
       -0.4472        -0.4472           -0.4472      -0.4472          2.2361       -0.4472      1.5667  -1.0  1.2637




# Calculo de los k-vecinos
- calcular las distancia 
-calcular la funcion knn


In [20]:
X_knn = X_transformado
y_knn = y.values
def distancia_euclidea(a, b):
    return np.sqrt(np.sum((a - b) ** 2))


def knn_predict(nuevo_paciente, k=3):

    # 1. Convertir a DataFrame
    nuevo_df = pd.DataFrame([nuevo_paciente])

    # 2. Transformar (MISMO ORDEN)
    nuevo_transformado = pipe_maestro.transform(nuevo_df)

    # 3. Calcular distancias
    distancias = []

    for i in range(len(X_knn)):
        dist = distancia_euclidea(nuevo_transformado[0], X_knn[i])
        distancias.append((i, dist, y_knn[i]))

    # 4. ORDENAMIENTO (AQUÍ ESTÁ LO QUE TE FALTABA ENTENDER)
    distancias.sort(key=lambda x: x[1])  
    # 👉 Ordena por la distancia (posición 1 de la tupla)

    # 5. Vecinos
    vecinos = distancias[:k]

    # 6. Mostrar
    print("\n--- DISTANCIAS ORDENADAS ---")
    for i, d, label in distancias:
        print(f"Paciente {i} → {round(d,4)} → {label}")

    print("\n--- VECINOS (k=3) ---")
    for i, d, label in vecinos:
        print(f"Paciente {i} → {round(d,4)} → {label}")

    # 7. Votación
    votos = [label for _, _, label in vecinos]
    prediccion = max(set(votos), key=votos.count)

    print("\n--- PREDICCIÓN ---")
    print("Diabetes:", "SI" if prediccion == 1 else "NO")

    return prediccion

def ingresar_paciente():
    print("=== INGRESO DE NUEVO PACIENTE ===")
    
    sexo = int(input("Ingrese sexo (1 o 2): "))
    ciudad = input("Ingrese ciudad: ")
    colesterol = input("Ingrese colesterol (bajo, medio, alto, muy alto): ")
    edad = int(input("Ingrese edad: "))
    
    paciente = {
        "sexo": sexo,
        "ciudad": ciudad,
        "colesterol": colesterol,
        "edad": edad
    }
    
    return paciente

nuevo = ingresar_paciente()
knn_predict(nuevo, k=3)

=== INGRESO DE NUEVO PACIENTE ===

--- DISTANCIAS ORDENADAS ---
Paciente 0 → 2.585 → 0
Paciente 4 → 3.9358 → 0
Paciente 2 → 3.992 → 0
Paciente 1 → 4.3525 → 1
Paciente 3 → 4.8688 → 1
Paciente 5 → 5.4771 → 1

--- VECINOS (k=3) ---
Paciente 0 → 2.585 → 0
Paciente 4 → 3.9358 → 0
Paciente 2 → 3.992 → 0

--- PREDICCIÓN ---
Diabetes: NO


0